# Spotify Audio Analytics: Data Acquisition & Preprocessing Pipeline
**Project Focus:** Predicting Track Popularity from Audio Features  
**Module Alignment:** Course Outcome 2 (CO2 — Data Preparation, Sanitization & Feature Engineering)  

---

### Executive Overview
This notebook implements the data ingestion, cleaning, deduplication, outlier filtering, feature scaling, and transformation pipeline for the 114,000-track Spotify dataset. The pipeline standardizes acoustic signals and produces the baseline analytical dataset used by subsequent exploratory data analysis, predictive modeling, and dashboard modules.

**Primary Outputs:**
- `cleaned_tracks.csv`: Final sanitized feature-engineered dataset (76,103 observations × 43 columns, 0 missing values).
- `scaler.pkl`: Pre-fitted `StandardScaler` model artifact ensuring consistent normalization across downstream tasks.


### Environment Setup & Configuration
Import essential analytical packages and set pipeline parameters.

In [1]:
import os
import re
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

# Pipeline Parameters
INPUT_CSV = "dataset.csv"
OUTPUT_CSV = "cleaned_tracks.csv"
SCALER_FILE = "scaler.pkl"

# Cold-start exposure filter: removes unpromoted tracks with 0 popularity to eliminate exposure noise
DROP_ZERO_POPULARITY = True

print("Environment configured successfully.")

Environment configured successfully.


## 1. Dataset Ingestion & Index Sanitization
- Load raw Kaggle Spotify tracks dataset (`dataset.csv`).
- Drop redundant CSV serialization index column (`Unnamed: 0`).
- Audit dataset dimensions and attribute schema.

In [2]:
df = pd.read_csv(INPUT_CSV)
original_len = len(df)
print(f"Loaded raw dataset with shape: {df.shape}")

# Remove artifact index column from previous CSV export
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])
    print("Removed redundant index column 'Unnamed: 0'.")

print(f"Active columns ({len(df.columns)}):\n{df.columns.tolist()}")
print(f"\nInitial observation count: {len(df):,}")
df.head(3)

Loaded raw dataset with shape: (114000, 21)
Removed redundant index column 'Unnamed: 0'.
Active columns (20):
['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']

Initial observation count: 114,000


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.1430,0.0322,0.000001,0.358,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.166,1,-17.235,1,0.0763,0.9240,0.000006,0.101,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,0,-9.734,1,0.0557,0.2100,0.000000,0.117,0.120,76.332,4,acoustic


## 2. Exploratory Data Profiling
- Inspect missing values across all columns.
- Compute transposed summary statistics to examine central tendency, dispersion, and range.
- Examine musical genre diversity.

In [3]:
print("--- Missing Values Summary ---")
null_counts = df.isnull().sum()
missing_summary = null_counts[null_counts > 0]
print(missing_summary if not missing_summary.empty else "No missing values detected.")

print(f"\nDistinct genre categories: {df['track_genre'].nunique()}")
print(f"Current row count: {len(df):,}")
df.describe(include="all").T[["count", "mean", "std", "min", "50%", "max"]].dropna(how="all")

--- Missing Values Summary ---
artists       1
album_name    1
track_name    1
dtype: int64

Distinct genre categories: 114
Current row count: 114,000


,count,mean,std,min,50%,max
track_id,114000,NaN,NaN,NaN,NaN,NaN
artists,113999,NaN,NaN,NaN,NaN,NaN
album_name,113999,NaN,NaN,NaN,NaN,NaN
track_name,113999,NaN,NaN,NaN,NaN,NaN
popularity,114000.0,33.238535,22.305078,0.0,35.0,100.0
duration_ms,114000.0,228029.153114,107297.712645,0.0,212906.0,5237295.0
explicit,114000,NaN,NaN,NaN,NaN,NaN
danceability,114000.0,0.5668,0.173542,0.0,0.58,0.985
energy,114000.0,0.641383,0.251529,0.0,0.685,1.0
key,114000.0,5.30914,3.559987,0.0,5.0,11.0


## 3. Handling Missing Values
- **Descriptive Metadata:** Missing text identifiers (`artists`, `album_name`, `track_name`) are imputed with `'Unknown'` to preserve complete acoustic signal measurements.
- **Core Numerical Features:** Fundamental audio measurements and the target variable (`popularity`) cannot be synthetically imputed without introducing bias; verify completeness and drop if null.

In [4]:
step3_start = len(df)

# Impute non-critical metadata
missing_artists = df["artists"].isnull().sum()
missing_albums = df["album_name"].isnull().sum()
df["artists"] = df["artists"].fillna("Unknown")
df["album_name"] = df["album_name"].fillna("Unknown")
df["track_name"] = df["track_name"].fillna("Unknown")
print(f"Imputed missing text metadata with 'Unknown' (artists: {missing_artists}, album: {missing_albums}).")

# Audit core audio features
core_features = ["danceability", "energy", "tempo", "valence", "loudness", "acousticness", "popularity"]
df = df.dropna(subset=core_features)
step3_dropped = step3_start - len(df)
print(f"Audited core features {core_features}. Filtered {step3_dropped} rows.")
print(f"Row count after Step 3: {len(df):,}")

Imputed missing text metadata with 'Unknown' (artists: 1, album: 1).
Audited core features ['danceability', 'energy', 'tempo', 'valence', 'loudness', 'acousticness', 'popularity']. Filtered 0 rows.
Row count after Step 3: 114,000


## 4. Two-Tier Deterministic Deduplication
1. **Tier 1 (Exact Spotify URI Deduplication):** Removes identical track entries occurring across multi-genre playlist aggregations.
2. **Tier 2 (Release Version Normalization with Deterministic Tiebreaker):** Strips version suffixes (`- Remastered`, `- Live`, `- Radio Edit`), sorts by `["popularity", "track_id"]` (descending popularity with unique `track_id` tiebreaker), and retains the primary composition per artist.
   - *Reproducibility Assurance:* 997 duplicate groups share identical popularity scores. Incorporating `track_id` eliminates sorting instability across operating systems and Python environments.

In [5]:
# 4a. Exact track_id deduplication
step4a_start = len(df)
df = df.drop_duplicates(subset=["track_id"])
dropped_exact_id = step4a_start - len(df)
print(f"Tier 1: Dropped {dropped_exact_id:,} duplicate track_id entries.")
print(f"Observations after exact deduplication: {len(df):,}")

# 4b. Song version deduplication with deterministic tiebreaker
step4b_start = len(df)
df["track_name_clean"] = df["track_name"].str.replace(
    r"\s*-\s*(Remaster(ed)?|Live|Radio Edit).*", "", regex=True, case=False
).str.strip()

# Sort by popularity descending with track_id tiebreaker for bit-for-bit reproducibility
df = df.sort_values(["popularity", "track_id"], ascending=[False, True])
df = df.drop_duplicates(subset=["track_name_clean", "artists"], keep="first")
dropped_version_dups = step4b_start - len(df)
print(f"Tier 2: Dropped {dropped_version_dups:,} duplicate release variations.")
print(f"Row count after Step 4: {len(df):,}")

Tier 1: Dropped 24,259 duplicate track_id entries.
Observations after exact deduplication: 89,741


Tier 2: Dropped 8,708 duplicate release variations.
Row count after Step 4: 81,033


## 5. Domain Outlier Filtering
- **Duration Constraints:** Filter recordings with `duration_ms <= 30,000` ms (short non-musical sound effects and calibration samples).
- **Tempo Plausibility:** Filter recordings outside standard musical cadences `[30, 250]` BPM (algorithmic estimation artifacts).
- **Cold-Start Exposure Filtering:** Filter unpromoted tracks with `popularity == 0` to eliminate platform exposure bias from predictive models.

In [6]:
step5_start = len(df)

# 5a. Duration filtering
step5a_start = len(df)
df = df[df["duration_ms"] > 30_000]
dropped_duration = step5a_start - len(df)
print(f"Filtered {dropped_duration:,} recordings with duration <= 30,000 ms.")

# 5b. Tempo plausibility filtering
step5b_start = len(df)
df = df[df["tempo"].between(30, 250)]
dropped_tempo = step5b_start - len(df)
print(f"Filtered {dropped_tempo:,} recordings with tempo outside [30, 250] BPM.")

# 5c. Zero-popularity filtering
step5c_start = len(df)
if DROP_ZERO_POPULARITY:
    df = df[df["popularity"] > 0]
    dropped_zero_pop = step5c_start - len(df)
    print(f"[Filter Active] Filtered {dropped_zero_pop:,} unpromoted tracks with popularity == 0.")
else:
    dropped_zero_pop = 0
    print("[Filter Inactive] Retained tracks with popularity == 0.")

total_outliers_dropped = step5_start - len(df)
print(f"Row count after Step 5: {len(df):,} (Total outliers filtered: {total_outliers_dropped:,})")

Filtered 15 recordings with duration <= 30,000 ms.
Filtered 143 recordings with tempo outside [30, 250] BPM.
[Filter Active] Filtered 4,772 unpromoted tracks with popularity == 0.
Row count after Step 5: 76,103 (Total outliers filtered: 4,930)


## 6. Continuous Feature Standardization
- Standardize 9 continuous numeric features to zero mean and unit variance ($\mu = 0, \sigma = 1$) using `StandardScaler`.
- Append normalized features with suffix `_scaled`.
- **Serialization:** Persist the fitted scaler to `scaler.pkl` to prevent data leakage and guarantee consistent feature representation across downstream clustering and inference.

In [7]:
scale_cols = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]
print(f"Continuous features to standardize: {scale_cols}")

scaler = StandardScaler()
scaled_feature_names = [f"{col}_scaled" for col in scale_cols]
df[scaled_feature_names] = scaler.fit_transform(df[scale_cols])

# Serialize scaler artifact
joblib.dump(scaler, SCALER_FILE)
print(f"Serialized fitted StandardScaler to '{SCALER_FILE}'.")
print(f"Scaler parameters: n_features = {scaler.n_features_in_}, mean shape = {scaler.mean_.shape}")
print(f"Row count after Step 6: {len(df):,}")

Continuous features to standardize: ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


Serialized fitted StandardScaler to 'scaler.pkl'.
Scaler parameters: n_features = 9, mean shape = (9,)
Row count after Step 6: 76,103


## 7. Categorical Encoding & Multicollinearity Prevention
- Convert boolean indicator `explicit` to binary integer ($0 / 1$).
- One-hot encode musical `key` (12 pitch classes) and `mode` (Major/Minor) with **`drop_first=True`**.
- **Statistical Rationale:** Omitting the reference level prevents the dummy variable trap (perfect multicollinearity), ensuring the feature covariance matrix $(X^TX)$ is non-singular and invertible for Ordinary Least Squares (OLS) regression.

In [8]:
# 7a. Convert binary explicit indicator
df["explicit"] = df["explicit"].astype(int)
print("Converted 'explicit' indicator to binary integer (0 / 1).")

# 7b. One-hot encoding with drop_first=True
cols_before = len(df.columns)
df = pd.get_dummies(df, columns=["key", "mode"], prefix=["key", "mode"], drop_first=True, dtype=int)
dummy_cols = [c for c in df.columns if c.startswith("key_") or c.startswith("mode_")]
print(f"Generated {len(dummy_cols)} dummy variables with drop_first=True: {dummy_cols}")
print(f"Expanded total columns from {cols_before} to {len(df.columns)}.")
print(f"Row count after Step 7: {len(df):,}")

Converted 'explicit' indicator to binary integer (0 / 1).
Generated 12 dummy variables with drop_first=True: ['key_1', 'key_2', 'key_3', 'key_4', 'key_5', 'key_6', 'key_7', 'key_8', 'key_9', 'key_10', 'key_11', 'mode_1']
Expanded total columns from 30 to 40.
Row count after Step 7: 76,103


## 8. Domain-Specific Feature Engineering
1. **`energy_valence`** ($= 	ext{energy} 	imes 	ext{valence}$): Interaction term modeling intense positive emotionality.
2. **`tempo_bucket`**: Discretization of tempo into intuitive bins (`slow`: $\le 90$, `mid`: $90–130$, `fast`: $> 130$ BPM).
3. **`mood_score`** ($= 0.5 	imes 	ext{valence} + 0.3 	imes 	ext{energy} + 0.2 	imes 	ext{danceability}$): Composite valence-energy-rhythm index.

In [9]:
# 8a. Valence-energy interaction feature
df["energy_valence"] = df["energy"] * df["valence"]

# 8b. Categorical tempo discretization
df["tempo_bucket"] = pd.cut(df["tempo"], bins=[0, 90, 130, 300], labels=["slow", "mid", "fast"])

# 8c. Composite mood index
df["mood_score"] = 0.5 * df["valence"] + 0.3 * df["energy"] + 0.2 * df["danceability"]

print("Engineered features created:")
print(f"- energy_valence (range): [{df['energy_valence'].min():.4f}, {df['energy_valence'].max():.4f}]")
print(f"- tempo_bucket distribution:\n{df['tempo_bucket'].value_counts().to_dict()}")
print(f"- mood_score (range): [{df['mood_score'].min():.4f}, {df['mood_score'].max():.4f}]")
print(f"Row count after Step 8: {len(df):,}")

Engineered features created:
- energy_valence (range): [0.0000, 0.9721]
- tempo_bucket distribution:
{'mid': 36444, 'fast': 28609, 'slow': 11050}
- mood_score (range): [0.0107, 0.9405]
Row count after Step 8: 76,103


## 9. Dataset Integrity Verification & Export
- Verify completeness across all cells (`df.isnull().sum().sum() == 0`).
- Verify `pd.cut` tempo boundaries: confirm 100% of observations lie within $(0, 300]$ BPM, ensuring 0 missing values in `tempo_bucket`.
- Validate final column count arithmetic ($20 + 1 + 9 - 2 + 12 + 3 = 43$).
- Export sanitized DataFrame to `cleaned_tracks.csv`.

In [10]:
print("[STEP 9] Executing dataset integrity verification and export...")

# Integrity verification
total_nulls = df.isnull().sum().sum()
tempo_bucket_nulls = df["tempo_bucket"].isnull().sum()
tempo_min, tempo_max = df["tempo"].min(), df["tempo"].max()

print("-> Dataset Integrity Audit:")
print(f"   • Total missing values across all cells: {total_nulls}")
print(f"   • 'tempo_bucket' missing values (pd.cut boundary audit): {tempo_bucket_nulls}")
print(f"   • Validated tempo range: [{tempo_min:.3f}, {tempo_max:.3f}] BPM (inside boundary [0, 300])")
assert total_nulls == 0, f"Integrity Failure: Expected 0 null values, found {total_nulls}."
print("   • Status: PASSED (0 missing values across all 43 columns).")

# Export to CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"Successfully exported {len(df):,} observations to '{OUTPUT_CSV}'.")

print("=" * 75)
print("FINAL PIPELINE AUDIT SUMMARY")
print("=" * 75)
print(f"Raw Input Row Count:        {original_len:,}")
print(f"Final Cleaned Row Count:    {len(df):,}")
print(f"Total Filtered Rows:        {(original_len - len(df)):,} ({((original_len - len(df))/original_len)*100:.2f}%)")
print(f"Final Column Count:         {len(df.columns)} (20 raw + clean_name + 9 scaled - 2 key/mode + 12 dummies + 3 engineered)")
print(f"Total Null Values:          {total_nulls}")
print(f"Cleaned CSV File Size:      {os.path.getsize(OUTPUT_CSV) / (1024 * 1024):.2f} MB")
print(f"Scaler Artifact Size:       {os.path.getsize(SCALER_FILE) / 1024:.2f} KB")
print("=" * 75)

[STEP 9] Executing dataset integrity verification and export...
-> Dataset Integrity Audit:
   • Total missing values across all cells: 0
   • 'tempo_bucket' missing values (pd.cut boundary audit): 0
   • Validated tempo range: [30.322, 243.372] BPM (inside boundary [0, 300])
   • Status: PASSED (0 missing values across all 43 columns).


Successfully exported 76,103 observations to 'cleaned_tracks.csv'.
FINAL PIPELINE AUDIT SUMMARY
Raw Input Row Count:        114,000
Final Cleaned Row Count:    76,103
Total Filtered Rows:        37,897 (33.24%)
Final Column Count:         43 (20 raw + clean_name + 9 scaled - 2 key/mode + 12 dummies + 3 engineered)
Total Null Values:          0
Cleaned CSV File Size:      29.77 MB
Scaler Artifact Size:       1.16 KB


## Methodological Defense & Technical Reference

| Technical Question | Methodological Defense |
| :--- | :--- |
| **Why filter ~38,000 observations (33%)?** | 24,259 entries were duplicate track URIs across multi-genre playlists. 8,708 were redundant remaster/live releases of identical compositions. Removing duplicates plus non-musical artifacts (<30s, extreme tempos) and 4,772 unpromoted 0-popularity tracks prevents exposure bias, distortion of sample variance, and train-test data leakage. |
| **How was deduplication reproducibility guaranteed?** | 997 duplicate song groups tie on maximum popularity. Sorting on `['popularity', 'track_id']` (descending popularity with unique Spotify track URI as secondary key) resolves ties deterministically, eliminating sorting algorithm instability across operating systems. |
| **Why use `drop_first=True` during one-hot encoding?** | Omitting one reference category avoids the dummy variable trap (perfect multicollinearity) across 12 keys and 2 modes, ensuring the feature covariance matrix $(X^TX)$ is non-singular and strictly invertible for regression modeling. |
| **Why serialize `scaler.pkl` rather than refitting downstream?** | Normalization parameters ($\mu, \sigma$) must be computed once on the baseline data distribution. Reusing the serialized scaler across clustering and inference prevents data leakage and ensures uniform scale alignment. |
| **How is complete data integrity verified with `pd.cut`?** | The discretization bins were defined over $[0, 90, 130, 300]$ BPM. Because Step 5b filtered tempos outside $[30, 250]$ BPM, 100% of observations fall strictly within $(0, 300]$, guaranteeing 0 missing values in `tempo_bucket`. |
